# 07: Random Forest

## 1. Imports

In [ ]:
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV,
    learning_curve,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

from sklearn.calibration import CalibrationDisplay

from src.config import PROJECT_ROOT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TRAIN_CLEAN_PATH,
    TRAIN_FEATURED_PATH,
    SPLIT_MANIFEST_PATH,
    MODELS_DIR,
    RESULTS_DIR,
    TARGET,
    ID_COLUMN,
    RANDOM_STATE,
)

pd.set_option("display.max_columns", 100)

## 2. Load Dataset

In [ ]:
clean = pd.read_csv(TRAIN_CLEAN_PATH)
featured = pd.read_csv(TRAIN_FEATURED_PATH)
manifest = pd.read_csv(SPLIT_MANIFEST_PATH)

train_ids = set(
    manifest.loc[
        manifest["Split"] == "train",
        ID_COLUMN,
    ]
)

holdout_ids = set(
    manifest.loc[
        manifest["Split"] == "test",
        ID_COLUMN,
    ]
)

def split_by_manifest(df):
    development = (
        df[df[ID_COLUMN].isin(train_ids)]
        .sort_values(ID_COLUMN)
        .reset_index(drop=True)
    )

    holdout = (
        df[df[ID_COLUMN].isin(holdout_ids)]
        .sort_values(ID_COLUMN)
        .reset_index(drop=True)
    )

    return development, holdout

clean_dev, clean_holdout = split_by_manifest(clean)
featured_dev, featured_holdout = split_by_manifest(featured)

print("Development rows:", len(clean_dev))
print("Holdout rows:", len(clean_holdout))
print("Development attrition rate:", f"{clean_dev[TARGET].mean():.1%}")
print("Holdout attrition rate:", f"{clean_holdout[TARGET].mean():.1%}")

## 3. Build Pipeline

In [ ]:
def build_random_forest_pipeline(
    X,
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features="sqrt",
    class_weight=None,
):
    categorical_columns = [
        column for column in X.columns
        if not pd.api.types.is_numeric_dtype(X[column])
    ]

    numeric_columns = [
        column for column in X.columns
        if column not in categorical_columns
    ]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ])

    classifier = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        max_features=max_features,
        class_weight=class_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])

## 4. Compare Features (Original v. Engineered)

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

comparison_rows = []

for feature_set_name, development_df in [
    ("Original cleaned features", clean_dev),
    ("Engineered candidate features", featured_dev),
]:
    X = development_df.drop(columns=[TARGET, ID_COLUMN])
    y = development_df[TARGET]

    pipeline = build_random_forest_pipeline(X)

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring={
            "roc_auc": "roc_auc",
            "pr_auc": "average_precision",
        },
        n_jobs=-1,
    )

    comparison_rows.append({
        "feature_set": feature_set_name,
        "cv_roc_auc_mean": scores["test_roc_auc"].mean(),
        "cv_roc_auc_std": scores["test_roc_auc"].std(),
        "cv_pr_auc_mean": scores["test_pr_auc"].mean(),
        "cv_pr_auc_std": scores["test_pr_auc"].std(),
    })

feature_comparison = pd.DataFrame(comparison_rows)

feature_comparison.round(4)

In [ ]:
plot_data = feature_comparison.set_index("feature_set")[
    ["cv_roc_auc_mean", "cv_pr_auc_mean"]
]

plot_data.plot(
    kind="bar",
    figsize=(8, 5),
)

plt.title("Random Forest: Feature Set CV Comparison")
plt.ylabel("Mean cross-validation score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
selected_name = (
    feature_comparison
    .sort_values("cv_pr_auc_mean", ascending=False)
    .iloc[0]["feature_set"]
)

print("Selected feature set:", selected_name)

if selected_name == "Engineered candidate features":
    development = featured_dev
    holdout = featured_holdout
else:
    development = clean_dev
    holdout = clean_holdout

X_dev = development.drop(columns=[TARGET, ID_COLUMN])
y_dev = development[TARGET]

X_holdout = holdout.drop(columns=[TARGET, ID_COLUMN])
y_holdout = holdout[TARGET]

## 5. Tune Model

In [ ]:
pipeline = build_random_forest_pipeline(X_dev)

param_distributions = {
    "classifier__n_estimators": [
        200,
        300,
        500,
        700,
    ],
    "classifier__max_depth": [
        None,
        4,
        6,
        8,
        12,
    ],
    "classifier__min_samples_leaf": [
        1,
        2,
        4,
        6,
        10,
    ],
    "classifier__min_samples_split": [
        2,
        4,
        8,
        12,
        20,
    ],
    "classifier__max_features": [
        "sqrt",
        "log2",
        0.5,
        None,
    ],
    "classifier__class_weight": [
        None,
        "balanced",
        "balanced_subsample",
    ],
}

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring={
        "pr_auc": "average_precision",
        "roc_auc": "roc_auc",
        "balanced_accuracy": "balanced_accuracy",
        "recall": "recall",
        "f1": "f1",
    },
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    return_train_score=True,
    verbose=1,
)

search.fit(X_dev, y_dev)

print("Best parameters:")
print(search.best_params_)
print()
print("Best CV PR-AUC:", round(search.best_score_, 4))

In [ ]:
tuning_results = pd.DataFrame(search.cv_results_)

tuning_view = tuning_results[[
    "param_classifier__n_estimators",
    "param_classifier__max_depth",
    "param_classifier__min_samples_leaf",
    "param_classifier__min_samples_split",
    "param_classifier__max_features",
    "param_classifier__class_weight",
    "mean_train_pr_auc",
    "mean_test_pr_auc",
    "std_test_pr_auc",
    "mean_test_roc_auc",
    "mean_test_balanced_accuracy",
    "mean_test_recall",
    "mean_test_f1",
    "rank_test_pr_auc",
]].copy()

tuning_view["overfit_gap"] = (
    tuning_view["mean_train_pr_auc"]
    - tuning_view["mean_test_pr_auc"]
)

tuning_view = tuning_view.sort_values(
    "rank_test_pr_auc"
)

tuning_view.head(10).round(4)

In [ ]:
top_tuning = tuning_view.head(10).copy()

def configuration_label(row):
    return (
        f"depth={row['param_classifier__max_depth']}, "
        f"leaf={row['param_classifier__min_samples_leaf']}, "
        f"features={row['param_classifier__max_features']}"
    )

top_tuning["configuration"] = top_tuning.apply(
    configuration_label,
    axis=1,
)

top_tuning = top_tuning.sort_values(
    "mean_test_pr_auc",
    ascending=True,
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_tuning["configuration"].astype(str),
    pd.to_numeric(top_tuning["mean_test_pr_auc"]),
)

plt.xlabel("Mean CV PR-AUC")
plt.title("Top Random Forest Configurations")
plt.tight_layout()
plt.show()

## 6. Overfitting Check

In [ ]:
overfit_view = (
    tuning_view
    .head(15)
    .copy()
)

overfit_view["configuration"] = overfit_view.apply(
    configuration_label,
    axis=1,
)

overfit_view = overfit_view.sort_values(
    "overfit_gap",
    ascending=True,
)

plt.figure(figsize=(10, 7))

plt.barh(
    overfit_view["configuration"].astype(str),
    overfit_view["overfit_gap"],
)

plt.xlabel("Train PR-AUC minus validation PR-AUC")
plt.title("Random Forest Overfitting Gap")
plt.tight_layout()
plt.show()

In [ ]:
best_row = tuning_view.iloc[0]

print(
    "Best configuration training PR-AUC:",
    round(best_row["mean_train_pr_auc"], 4),
)

print(
    "Best configuration validation PR-AUC:",
    round(best_row["mean_test_pr_auc"], 4),
)

print(
    "Overfit gap:",
    round(best_row["overfit_gap"], 4),
)

## 7. Learning Curve

In [ ]:
best_model = search.best_estimator_

train_sizes, train_scores, validation_scores = learning_curve(
    best_model,
    X_dev,
    y_dev,
    cv=cv,
    scoring="average_precision",
    train_sizes=np.linspace(0.25, 1.0, 5),
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
validation_mean = validation_scores.mean(axis=1)

plt.figure(figsize=(8, 5))

plt.plot(
    train_sizes,
    train_mean,
    marker="o",
    label="Training PR-AUC",
)

plt.plot(
    train_sizes,
    validation_mean,
    marker="o",
    label="Validation PR-AUC",
)

plt.xlabel("Training examples")
plt.ylabel("PR-AUC")
plt.title("Random Forest Learning Curve")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Evaluate

In [ ]:
holdout_probability = best_model.predict_proba(
    X_holdout
)[:, 1]

holdout_prediction = best_model.predict(
    X_holdout
)

metrics = pd.DataFrame([{
    "accuracy": accuracy_score(
        y_holdout,
        holdout_prediction,
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_holdout,
        holdout_prediction,
    ),
    "precision": precision_score(
        y_holdout,
        holdout_prediction,
        zero_division=0,
    ),
    "recall": recall_score(
        y_holdout,
        holdout_prediction,
        zero_division=0,
    ),
    "f1": f1_score(
        y_holdout,
        holdout_prediction,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        y_holdout,
        holdout_probability,
    ),
    "pr_auc": average_precision_score(
        y_holdout,
        holdout_probability,
    ),
    "brier_score": brier_score_loss(
        y_holdout,
        holdout_probability,
    ),
}])

metrics.round(4)

## 9. Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_holdout,
    holdout_prediction,
)

plt.title("Random Forest — Confusion Matrix")
plt.tight_layout()
plt.show()

## 10. ROC Curve

In [ ]:
RocCurveDisplay.from_predictions(
    y_holdout,
    holdout_probability,
    name="Random Forest",
)

plt.title("Random Forest — ROC Curve")
plt.tight_layout()
plt.show()

## 11. Precision-Recall Curve

In [ ]:
PrecisionRecallDisplay.from_predictions(
    y_holdout,
    holdout_probability,
    name="Random Forest",
)

plt.title("Random Forest — Precision–Recall Curve")
plt.tight_layout()
plt.show()

## 12. Predicted Probability Distributions

In [ ]:
probability_frame = pd.DataFrame({
    "Actual": y_holdout.values,
    "Probability": holdout_probability,
})

plt.figure(figsize=(8, 5))

for actual_value in sorted(
    probability_frame["Actual"].unique()
):
    values = probability_frame.loc[
        probability_frame["Actual"] == actual_value,
        "Probability",
    ]

    plt.hist(
        values,
        bins=15,
        alpha=0.6,
        label=f"Actual = {actual_value}",
    )

plt.xlabel("Predicted attrition probability")
plt.ylabel("Employees")
plt.title("Random Forest — Probability Distribution")
plt.legend()
plt.tight_layout()
plt.show()

## 13. Calibration

In [ ]:
CalibrationDisplay.from_predictions(
    y_holdout,
    holdout_probability,
    n_bins=6,
    strategy="quantile",
    name="Random Forest",
)

plt.title("Random Forest — Calibration Curve")
plt.tight_layout()
plt.show()

## 14. Feature Importance

In [ ]:
preprocessor = best_model.named_steps["preprocessor"]
classifier = best_model.named_steps["classifier"]

feature_names = [
    name.split("__", 1)[-1]
    for name in preprocessor.get_feature_names_out()
]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": classifier.feature_importances_,
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False,
)

feature_importance.head(20)

In [ ]:
top_features = (
    feature_importance
    .head(20)
    .sort_values("importance")
)

plt.figure(figsize=(9, 8))

plt.barh(
    top_features["feature"],
    top_features["importance"],
)

plt.xlabel("Feature importance")
plt.title("Random Forest — Top 20 Feature Importances")
plt.tight_layout()
plt.show()

## 15. Compare against earlier models

In [ ]:
comparison_rows = []

baseline_path = RESULTS_DIR / "baseline_results.csv"
logistic_path = RESULTS_DIR / "logistic_regression_results.csv"

if baseline_path.exists():
    baseline = pd.read_csv(baseline_path)

    prior = baseline.loc[
        baseline["model"] == "Dummy - Prior"
    ].iloc[0]

    comparison_rows.append({
        "model": "Dummy - Prior",
        "roc_auc": prior["roc_auc"],
        "pr_auc": prior["pr_auc"],
        "balanced_accuracy": prior["balanced_accuracy"],
        "recall": prior["recall"],
    })

if logistic_path.exists():
    logistic = pd.read_csv(logistic_path).iloc[0]

    comparison_rows.append({
        "model": "Logistic Regression",
        "roc_auc": logistic["roc_auc"],
        "pr_auc": logistic["pr_auc"],
        "balanced_accuracy": logistic["balanced_accuracy"],
        "recall": logistic["recall"],
    })

comparison_rows.append({
    "model": "Random Forest",
    "roc_auc": metrics["roc_auc"].iloc[0],
    "pr_auc": metrics["pr_auc"].iloc[0],
    "balanced_accuracy": metrics[
        "balanced_accuracy"
    ].iloc[0],
    "recall": metrics["recall"].iloc[0],
})

comparison = pd.DataFrame(comparison_rows)

comparison.set_index("model").T.plot(
    kind="bar",
    figsize=(10, 5),
)

plt.title("Random Forest vs Earlier Models")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

comparison.round(4)

## 16. Save Results

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

feature_comparison.to_csv(
    RESULTS_DIR / "random_forest_feature_set_comparison.csv",
    index=False,
)

tuning_view.to_csv(
    RESULTS_DIR / "random_forest_tuning_results.csv",
    index=False,
)

model_results = metrics.copy()

model_results.insert(
    0,
    "model",
    "Random Forest",
)

model_results["feature_set"] = selected_name
model_results["cv_pr_auc"] = search.best_score_

for parameter, value in search.best_params_.items():
    clean_parameter = parameter.replace(
        "classifier__",
        "best_",
    )
    model_results[clean_parameter] = str(value)

model_results.to_csv(
    RESULTS_DIR / "random_forest_results.csv",
    index=False,
)

holdout_predictions = pd.DataFrame({
    ID_COLUMN: holdout[ID_COLUMN].values,
    "Actual": y_holdout.values,
    "Predicted": holdout_prediction,
    "AttritionProbability": holdout_probability,
})

holdout_predictions.to_csv(
    RESULTS_DIR / "random_forest_holdout_predictions.csv",
    index=False,
)

feature_importance.to_csv(
    RESULTS_DIR / "random_forest_feature_importance.csv",
    index=False,
)

joblib.dump(
    best_model,
    MODELS_DIR / "random_forest_pipeline.joblib",
)

print("Results saved to:", RESULTS_DIR)
print("Model saved to:", MODELS_DIR)